<a href="https://colab.research.google.com/github/EmanAbuhamra/MedBot/blob/main/Medical_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 6 Challenge - Building a Medical Chatbot
In this challenge, we will fine-tune a pretrained conversational model using a Q&A dataset to obtain a custom version of MedBot. After importing the pretrained model and dataset, we will apply techniques like Bits & Bytes quantization and LoRA fine-tuning to work around GPU limitations in google collab. Lastly, we will test the model by using it to generate answers to  medical questions using prompts from the training and testing data, to evaluate the model and note the final remarks.

The short video (x2 speed) showing the bot answering questions can be found here: https://youtu.be/n5EFfY911t4

## Importing Dataset and Model

In [ ]:
from datasets import load_dataset

ds = load_dataset("medalpaca/medical_meadow_wikidoc_patient_information")

In [ ]:
!pip install -U transformers
!pip install -q bitsandbytes accelerate transformers peft
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00


In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# load model

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "EleutherAI/gpt-neox-20b"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

Loading weights:   0%|          | 0/532 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
# dataset structure
print(ds)
print(ds["train"].column_names)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 5942
    })
})
['input', 'output', 'instruction']
{'input': 'What are the symptoms of Allergy?', 'output': 'Allergy symptoms vary, but may include:\nBreathing problems (coughing, shortness of breath) Burning, tearing, or itchy eyes Conjunctivitis (red, swollen eyes) Coughing Diarrhea Headache Hives Itching of the nose, mouth, throat, skin, or any other area Runny nose Skin rashes Stomach cramps Vomiting Wheezing\nWhat part of the body is contacted by the allergen plays a role in the symptoms you develop. For example:\nAllergens that are breathed in often cause a stuffy nose, itchy nose and throat, mucus production, cough, or wheezing. Allergens that touch the eyes may cause itchy, watery, red, swollen eyes. Eating something you are allergic to can cause nausea, vomiting, abdominal pain, cramping, diarrhea, or a severe, life-threatening reaction. Allergens that touch the skin can c

After importing the dataset, we find that it consists of 5942 observations and 3 columns: input, output and instruction. For example, the input column asks "What are the symptoms of Allergy?", the output column: "Allergy symptoms vary, but may include:\nBreathing problems...", and the instruction column: "Answer this question truthfully". Thus, each observation is a medical question paired with its answer. Next, we will preprocess this data by splitting it into training and testing data to train our pretrained model.

## Preprocessing

In [ ]:
# Set padding token
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Split dataset
ds_split = ds["train"].train_test_split(test_size=0.1, seed=42)

train_ds = ds_split["train"]
val_ds = ds_split["test"]

In [ ]:
def format_prompt(example):
    instruction = example["instruction"]
    input_text = example["input"]
    output = example["output"]

    if input_text.strip():
        text = f"""### Instruction:
{instruction}

### Input:
{input_text}

### Response:
{output}{tokenizer.eos_token}"""
    else:
        text = f"""### Instruction:
{instruction}

### Response:
{output}{tokenizer.eos_token}"""

    return {"text": text}

train_ds = train_ds.map(format_prompt)
val_ds = val_ds.map(format_prompt)

Map:   0%|          | 0/5347 [00:00<?, ? examples/s]

Map:   0%|          | 0/595 [00:00<?, ? examples/s]

In [ ]:
# tokenize
max_length = 512

def tokenize(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_train = train_ds.map(tokenize, batched=True)
tokenized_val = val_ds.map(tokenize, batched=True)

Map:   0%|          | 0/5347 [00:00<?, ? examples/s]

Map:   0%|          | 0/595 [00:00<?, ? examples/s]

In [ ]:
# remove extra columns
tokenized_train = tokenized_train.remove_columns(train_ds.column_names)
tokenized_val = tokenized_val.remove_columns(val_ds.column_names)

tokenized_train.set_format("torch")
tokenized_val.set_format("torch")

We first split the data into training, validation and testing. Next, we reformatted the observations into structured prompts to provide the model with a consistent conversational format for learning. Finally, we tokenized the formatted text into numerical input so the causal language model could learn to predict the next token during fine-tuning.


## Fine-Tuning
In this step, we will use LoRA fine-tuning as our PEFT choice as it trains only a small number of parameters instead of all model parameters, as it will allow us to work within the limitations of the GPU and speed up training.



In [ ]:
from peft import prepare_model_for_kbit_training

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "query_key_value"
    ]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 30,277,632 || all params: 20,584,845,312 || trainable%: 0.1471


After applying LoRA, only 30,277,632 of the model's 20.59 billion parameters remain trainable, representing just 0.1471% of the total parameters. The remaining 99.85% of the pretrained weights are frozen. This demonstrates the efficiency of parameter-efficient fine-tuning, as the model can be adapted to the medical Q&A task while requiring significantly less memory and computational resources than full fine-tuning.

## Configuration of Training Parameters


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./medbot-gpt-neox-20b-lora",
    max_steps=20,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    fp16=True,
    optim="paged_adamw_8bit",
    logging_steps=5,
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False
)

In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

# Trainer

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer
)

In [ ]:
# Disable cache for GPU storage
model.config.use_cache = False

In [ ]:
# Enable gradient checkpointing
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

In [ ]:
trainer.train()

Step,Training Loss
5,1.772540
10,1.683732
15,1.645506
20,1.579311


TrainOutput(global_step=20, training_loss=1.6702722311019897, metrics={'train_runtime': 1742.7164, 'train_samples_per_second': 0.184, 'train_steps_per_second': 0.011, 'total_flos': 1.993112725487616e+16, 'train_loss': 1.6702722311019897, 'epoch': 0.05984664297737049})

The model was trained for a single epoch because GPT-NeoX-20B is already pretrained on a large corpus of text, and the objective was to adapt it to the medical question–answering task using LoRA rather than train it from scratch. Given the computational cost of fine-tuning a 20-billion-parameter model, one epoch provides a practical balance between learning task-specific patterns and maintaining feasible training time on limited hardware.

The model completed the specified 20 training steps successfully, resulting in an average training loss of 1.67. Training took approximately 29 minutes on a T4 GPU, reflecting the high computational cost of fine-tuning a 20-billion-parameter language model even when using LoRA and 4-bit quantization. Since training was limited by the max_steps parameter, only about 6% of one epoch was completed. Despite the short training duration, the model was successfully adapted and is ready for qualitative evaluation using medical question–answer pairs.

## Evaluation


In [ ]:
import math

training_loss = 1.6702722311019897
train_perplexity = math.exp(training_loss)

print("Training Loss:", training_loss)
print("Training Perplexity:", train_perplexity)

Training Loss: 1.6702722311019897
Training Perplexity: 5.313614131334186


Due to GPU memory constraints associated with fine-tuning the 20-billion-parameter GPT-NeoX model, a separate evaluation on the validation set was not performed. Instead, the model will be evaluated using the final training loss, the corresponding training perplexity, and qualitative testing on both dataset examples and unseen medical questions.

The model successfully completed the fine-tuning process using LoRA and 4-bit quantization. After 20 training steps, it achieved a training loss of 1.67, corresponding to a training perplexity of approximately 5.31. The relatively low perplexity suggests that the model learned to predict the next tokens in the medical question–answer pairs with reasonable confidence despite the limited training. Training took approximately 29 minutes on a T4 GPU, highlighting the computational demands of fine-tuning a 20-billion-parameter language model. Since training was intentionally limited to 20 steps (about 6% of one epoch) due to hardware and time constraints, the model was only partially adapted to the dataset. Nevertheless, the results indicate that the fine-tuning pipeline was successfully implemented and the model is suitable for qualitative evaluation on medical questions.

While additional training steps or a complete epoch could potentially improve the model's performance, they would require substantially more computational resources and training time. The chosen configuration represents a practical compromise between model adaptation and the available hardware limitations.

## Testing

In [ ]:
def generate_answer(question):
    prompt = f"""### Instruction:
Answer the following medical question clearly and safely.

### Input:
{question}

### Response:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    answer = tokenizer.decode(output[0], skip_special_tokens=True)
    return answer.split("### Response:")[-1].strip()

In [ ]:
for i in range(3):
    question = ds["train"][i]["input"]

    if question.strip() == "":
        question = ds["train"][i]["instruction"]

    print("Question:", question)
    print("Model Answer:", generate_answer(question))
    print("Reference Answer:", ds["train"][i]["output"])
    print("-" * 80)

Question: What are the symptoms of Allergy?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model Answer: Symptoms of Allergy: The signs and symptoms of Allergy vary from person to person. Some people have mild symptoms. Other people have severe symptoms. Symptoms may include:

* Itching
* Skin rash
* Swelling
* Breathing problems
* Sneezing
* Fever
* Vomiting
* Diarrhea
* Cough
* Nausea
* Fatigue
* Headache
* Dizziness
* Shortness of breath
* Wheezing
* Joint pain
* Itching and burning of the skin
Reference Answer: Allergy symptoms vary, but may include:
Breathing problems (coughing, shortness of breath) Burning, tearing, or itchy eyes Conjunctivitis (red, swollen eyes) Coughing Diarrhea Headache Hives Itching of the nose, mouth, throat, skin, or any other area Runny nose Skin rashes Stomach cramps Vomiting Wheezing
What part of the body is contacted by the allergen plays a role in the symptoms you develop. For example:
Allergens that are breathed in often cause a stuffy nose, itchy nose and throat, mucus production, cough, or wheezing. Allergens that touch the eyes may caus

In [ ]:
new_questions = [
    "When to seek urgent medical care when I have Allergy?",
    "What are common symptoms of diabetes?",
    "How can high blood pressure be managed?",
    "What should I do if I have a fever?",
    "What are common causes of headaches?",
    "When should someone seek medical help for chest pain?"
]

for q in new_questions:
    print("Question:", q)
    print("Model Answer:", generate_answer(q))
    print("-" * 80)

Question: When to seek urgent medical care when I have Allergy?
Model Answer: If you have symptoms of an allergic reaction, see a doctor as soon as possible. Most people can take an antihistamine or epinephrine to treat an allergic reaction. If you have a severe reaction, you may need to be treated in the hospital.
You can take an antihistamine to treat an allergic reaction if you have symptoms of an allergic reaction, such as:

Severe allergic reactions to bee stings or other insect stings can be life-threatening. If you have symptoms of an allergic reaction, see a doctor as soon as possible.

If you have
--------------------------------------------------------------------------------
Question: What are common symptoms of diabetes?
Model Answer: Symptoms of diabetes include:

Fatigue (extreme tiredness)

Excessive thirst

Frequent urination

Increased appetite

Blurred vision

Increased hunger

Weight loss

Constipation

Nausea

Dry mouth

Vaginal itching

Dry skin

More frequent infe

## Concluding Remarks

Several libraries and frameworks were selected to build the medical chatbot efficiently. The Transformers library by Hugging Face was used to load the pretrained GPT-NeoX-20B model and tokenizer, while the Datasets library simplified loading, preprocessing, splitting, and tokenizing the medical question–answer dataset. PEFT (Parameter-Efficient Fine-Tuning) was chosen to implement LoRA, enabling efficient fine-tuning by updating only a small fraction of the model's parameters. BitsAndBytes was used for 4-bit quantization to significantly reduce GPU memory usage, making it possible to fine-tune the large model on limited hardware. Finally, PyTorch provided the underlying deep learning framework, and the Trainer API from Transformers streamlined the training process with minimal custom code.

The chatbot was tested using both examples from the dataset and new unseen medical questions. Overall, the model was able to generate medically relevant responses in some cases, showing that the fine-tuning pipeline successfully adapted the pretrained model toward medical question answering. However, because training was limited to only 20 steps due to GPU constraints, some answers may still be vague, incomplete, or overly general. There is also a risk of hallucination, where the model may generate information that sounds medically plausible but is not fully accurate. The model could be improved by training for more steps, using a larger portion of the dataset, evaluating on a validation set, improving prompt formatting, and applying stricter medical safety instructions during generation.